> **⚠️ Run this inside the NVIDIA container on a g5 (A10G).** The TensorRT engine only loads in the `nvcr.io/nvidia/pytorch:22.07-py3` runtime (CUDA 11.7.1 / TensorRT 8.4.1.5) — it will **not** run in a host venv.

Follow the repo **README → “Runtime setup”** to start the container + a Jupyter server inside it, then point this notebook's kernel at it (kernel picker → **Existing Jupyter Server** → `http://localhost:8888`). Set `BRIA_API_TOKEN` + `HF_TOKEN` (with approved access to the gated `briaai/increase-resolution`) as env vars on the container.

# Increase Resolution — capabilities walkthrough

This notebook runs **Increase Resolution** end to end: install the **`increase-resolution`** package from **AWS CodeArtifact**, download the Bria super-resolution **TensorRT engine(s)**, then upscale an image **2×** and **4×** with the in-process tiling pipeline.

**Required environment (not flexible):** NVIDIA **A10** (Ampere), **CUDA 11.7.1**, **TensorRT 8.4.1.5** (exact patch), Python **3.10**. The exact CUDA 11.7.1 libraries are not pip-installable, so use the NVIDIA base image `nvcr.io/nvidia/pytorch:22.07-py3` (CUDA 11.7.1 + TensorRT 8.4.1.5) with Python 3.10. **Before running this notebook**, prepare the runtime as described in the README's *Runtime setup* section (base image + `unset PYTHONPATH`/`LD_LIBRARY_PATH`, the cuBLAS `LD_PRELOAD`); installing `increase-resolution[all]` (cell below) pulls the GPU runtime (torch cu117 + nvidia-tensorrt). A TensorRT engine only loads on the GPU/runtime it was built for.

**Prerequisites:** `BRIA_API_TOKEN` (for the CodeArtifact credential).

In [ ]:
from dotenv import load_dotenv

load_dotenv()  # loads BRIA_API_TOKEN (and friends) from a local .env file, if present

## 1. CodeArtifact token (Bria Engine)

Request a short-lived credential for the **`bria-increase-res`** repository.

In [ ]:
import os

import requests

BRIA_API_TOKEN = os.environ.get("BRIA_API_TOKEN")
if not BRIA_API_TOKEN:
    raise RuntimeError("Set BRIA_API_TOKEN before running this notebook.")

resp = requests.get(
    "https://engine.prod.bria-api.com/v2/auth/access/code_artifact",
    params={"repository": "bria-increase-res"},
    headers={"api_token": BRIA_API_TOKEN},
    timeout=60,
)
resp.raise_for_status()
result = resp.json()["result"]
CODE_ARTIFACT_PASSWORD = result["authorization_token"].strip()
print("CodeArtifact credential acquired. Expires:", result.get("expiration"))

## 2. Install `increase-resolution`

Uses the CodeArtifact token as the password for the `bria-increase-res` PyPI index (username `aws`). `torch` and `tensorrt==8.4.*` must already match your CUDA 11.7 environment.

In [ ]:
import subprocess
import sys
from urllib.parse import quote

CA = "https://aws:" + quote(CODE_ARTIFACT_PASSWORD, safe="") + "@bria-300465780738.d.codeartifact.us-east-1.amazonaws.com/pypi/bria-increase-res/simple/"
# [all] = full pipeline: pulls torch (cu117) + nvidia-tensorrt from their indexes + the rest.
subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "increase-resolution[all]",
        "huggingface_hub",
        "--extra-index-url",
        "https://download.pytorch.org/whl/cu117",
        "--extra-index-url",
        "https://pypi.ngc.nvidia.com",
        "--extra-index-url",
        CA,
    ]
)

## 3. Download the super-resolution engine(s)

Bria hosts the `.engine` files on the Hugging Face repo **`briaai/increase-resolution`** (private — request/receive access from Bria, then set `HF_TOKEN`). `hf_hub_download` fetches them into the local HF cache.

**Note:** the engines are TensorRT builds tied to the exact GPU + CUDA/TensorRT runtime documented above; they only load on a matching environment.

In [ ]:
import os

from huggingface_hub import hf_hub_download

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN for Hugging Face access to briaai/increase-resolution.")

# Bria hosts the super-resolution TensorRT engines on the private HF repo briaai/increase-resolution
# (request/receive access from Bria). hf_hub_download fetches them into the local HF cache.
WEIGHTS_REPO = "briaai/increase-resolution"
engine_paths = {}
for scale in (2, 4):
    engine_paths[scale] = hf_hub_download(repo_id=WEIGHTS_REPO, filename=f"increase_resolution{scale}.engine", token=hf_token)
    print(f"x{scale} engine:", engine_paths[scale])

## 4. Build configuration and start the pipeline

In [ ]:
import torch
from IPython.display import display

from increase_resolution import IncreaseResolution, IncreaseResolutionConfig, IncreaseResolutionInput

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for Increase Resolution.")

config = IncreaseResolutionConfig(engine_paths={int(k): v for k, v in engine_paths.items()})
pipeline = IncreaseResolution(config=config)
pipeline.setup()
print("Pipeline setup complete.")

## 5. Upscale a sample image (2× and 4×)

In [ ]:
import time
from pathlib import Path

Path("outputs").mkdir(exist_ok=True)

SAMPLE = "https://bria-test-images.s3.us-east-1.amazonaws.com/highway.jpg"

t0 = time.time()
out2 = pipeline.execute(IncreaseResolutionInput(image=SAMPLE, desired_increase=2))
print(f"2x: {out2.image.size} in {time.time() - t0:.1f}s")
out2.image.save("outputs/upscaled_x2.png")
display(out2.image)

In [ ]:
t0 = time.time()
out4 = pipeline.execute(IncreaseResolutionInput(image=SAMPLE, desired_increase=4))
print(f"4x: {out4.image.size} in {time.time() - t0:.1f}s")
out4.image.save("outputs/upscaled_x4.png")
display(out4.image)

## 6. Cleanup

In [ ]:
pipeline.cleanup()
print("Cleanup done.")